# 03 — Shared input contract for fair comparison

This notebook confirms the invariant that makes the benchmark a fair comparison: every architecture receives the exact same input tensor, the same number of input channels, the same spatial size, and is asked for the same number of output channels. Only the model differs between trials.

Modules exercised: `configuration/benchmark_config.py` (`SizeMatchConfig.in_channels`, `TrainingQueueConfig.patch_size`, `BenchmarkConfig.n_gaussians`) and `pipelines/benchmark_pipeline/sizing.py` (`SizeMatcher` derives `in_channels`, `out_channels`, `image_size` from these config fields). We reproduce that derivation and feed several backbones one shared batch.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import warnings
warnings.filterwarnings("ignore")

import numpy             as np
import torch
import matplotlib.pyplot as plt

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(False)

DEVICE = torch.device("cpu")

plt.rcParams.update({
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "axes.axisbelow" : True,
})

print("bootstrap ready  —  torch", torch.__version__, " device", DEVICE)


## Deriving the shared contract from the config

`SizeMatcher.__init__` reads three values from `BenchmarkConfig`: the input channel count, the output channel count (`n_gaussians * 3`), and the image size (`patch_size[0]`). We read the same fields here so the contract used below is literally the pipeline's contract.

In [ ]:
from configuration.benchmark_config import BenchmarkConfig

config = BenchmarkConfig()

in_channels  = config.size_match.in_channels
out_channels = config.n_gaussians * 3
image_size   = config.training.patch_size[0]

print(f"in_channels  (SizeMatchConfig.in_channels)      : {in_channels}")
print(f"out_channels (n_gaussians * 3)                  : {out_channels}")
print(f"image_size   (TrainingQueueConfig.patch_size[0]): {image_size}")

## One shared synthetic batch

A single seeded batch is built to the contract and frozen. The same object is handed to every backbone below; there is no per-model preprocessing or reshaping.

In [ ]:
BATCH = 2

shared_batch = torch.randn(BATCH, in_channels, image_size, image_size, device=DEVICE)
shared_batch.requires_grad_(False)

input_signature = (tuple(shared_batch.shape), float(shared_batch.mean()), float(shared_batch.std()))
print("shared batch shape    :", input_signature[0])
print("shared batch mean/std :", round(input_signature[1], 5), round(input_signature[2], 5))

## Feeding the same batch to several backbones

We pick a handful of structurally different backbones (a plain CNN U-Net, a residual variant, a transformer encoder, and a pyramid network), reduce each to a small width, and run the shared batch through all of them. Before each forward pass we assert the input is byte-for-byte the one we created.

In [ ]:
from models import get_model
from pipelines.benchmark_pipeline.sizing import WidthScaler, _IMAGE_SIZE_MODELS

selected = ["unet", "resunet", "segformer", "fpn"]
scaler   = WidthScaler()
outputs  = {}

for name in selected:
    current_signature = (tuple(shared_batch.shape), float(shared_batch.mean()), float(shared_batch.std()))
    assert current_signature == input_signature, "shared batch was mutated between models"

    config_obj = scaler.scaled_config(name, 0.2)
    overrides  = {"in_channels": in_channels, "out_channels": out_channels}
    if name in _IMAGE_SIZE_MODELS:
        overrides["image_size"] = image_size

    model, _ = get_model(name, config=config_obj, **overrides)
    model.eval()
    with torch.no_grad():
        outputs[name] = model(shared_batch).detach()
    del model

for name, y in outputs.items():
    print(f"{name:12s} output {tuple(y.shape)}  mean {float(y.mean()):+.4f}")

shapes = {tuple(y.shape) for y in outputs.values()}
assert len(shapes) == 1, "backbones disagreed on output shape under shared contract"
print("\nall outputs share one shape:", shapes.pop())

## Visual confirmation

The left panel shows one channel of the shared input, identical for every model. The remaining panels show the same channel of each model's output. Inputs are indistinguishable across models (they are the same tensor); outputs differ because the architectures differ, which is exactly what a fair benchmark isolates.

In [ ]:
channel = 0
fig, axes = plt.subplots(1, len(selected) + 1, figsize=(3.0 * (len(selected) + 1), 3.2))

axes[0].imshow(shared_batch[0, channel].cpu().numpy(), cmap="viridis")
axes[0].set_title("shared input\nchannel 0")
axes[0].grid(False)

for ax, name in zip(axes[1:], selected):
    ax.imshow(outputs[name][0, channel].cpu().numpy(), cmap="magma")
    ax.set_title(f"{name}\noutput ch 0")
    ax.grid(False)

for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle("Same input, different backbones, identical output geometry")
fig.tight_layout()
plt.show()

## Expected visual outcome

The signature assertions hold (the shared batch is never mutated), all four backbones emit the same output shape, and the figure shows one identical input map beside four visibly different output maps of identical dimensions. This is the fairness contract made visible.